# Example notebook of a Variant QC analysis

The present notebook serves as a guide of how use the `IDEAL-GENOM-QC` library to perform a variant quality control. We intend to show a possible use, because each user can adapt it to its particular needs.

In this notebook the procedure to perform the variant quality control is more detailed so the user can get a deeper understanding of all the steps executed in this part of the pipeline.

Let us import the required libraries.

In [ ]:
import sys
import os

import pandas as pd

from pathlib import Path

# add parent directory to path
library_path = os.path.abspath('..')
if library_path not in sys.path:
    sys.path.append(library_path)

library_path = Path(library_path)

from ideal_genom.qc.variant_qc import VariantQC, VariantQCReport, VariantQCCleanUp

In the next cell the path variables associated with the project are set.

In [ ]:
DATA_PATH = library_path / 'ideal_genom' / 'data'

test_data = DATA_PATH / 'test_data'
inputData = test_data / 'inputData'
ouputData = test_data / 'outputData'

input_path = inputData
input_name = '1kG_phase3_GRCh38_updated'
output_path= ouputData
output_name= '1KG_GRCh38_variant_qc'

In the next cell we define a dictionary containing the parameters to execute the variant QC pipeline.

The explanation of the parameters is the following:

1. `chr_y`: identifier of Y chromosome in plink binary files.
2. `miss_data_rate`: Missing data rate threshold for variants.
3. `diff_genotype_rate`: P-value threshold for the case/control differential missingness test.
4. `geno`: Parameter `--geno` of **PLINK1.9**, maximum per-variant missing genotype rate.
5. `hwe`: Parameter `--hwe` of **PLINK1.9**. Note there is no separate HWE *check* step that produces its own fail list — the threshold is applied directly during the final filtering step (`execute_drop_variants`), together with `maf` and `geno`.
6. `maf`: Parameter `--maf` of **PLINK1.9**.

In [ ]:
variant_params = {
    'chr_y': 24,
    'miss_data_rate': 0.2,
    'diff_genotype_rate': 1e-5,
    'geno': 0.1,
    'maf': 5e-8,
    'hwe': 5e-8,
}

Initialize the class `VariantQC`.

In [ ]:
variant = VariantQC(
    input_path=input_path,
    input_name=input_name,
    output_path=output_path,
    output_name=output_name,
)

Execute the pipeline steps of the variant quality control. `execute_variant_qc_pipeline()` runs every QC step in order (missing data rate, case/control differential missingness, fail-variant aggregation), and drops the failing variants, producing cleaned `PLINK` files in `variant.clean_dir`.

The pipeline shells out to PLINK many times, which prints a lot of console text; we capture it into `variant_qc_log` to keep the notebook readable — run `variant_qc_log.show()` in a new cell if you need to inspect it.

In [ ]:
%%capture variant_qc_log
variant.execute_variant_qc_pipeline(variant_params=variant_params)

In [ ]:
print(f"Variant QC pipeline completed. Clean PLINK files written to: {variant.clean_dir}")

**Note:** `execute_variant_qc_pipeline()` already performs the full pipeline end-to-end, including aggregating all QC failures (`get_fail_variants()`) and removing them (`execute_drop_variants()`). The cells below are for *inspecting and customizing* the resulting reports — they do not need to repeat the failure aggregation or variant dropping.

In [ ]:
report = VariantQCReport(
    output_path=variant.plots_dir
)

In [ ]:
report.report_variant_qc(
    missing_data_rate_male  =variant.males_missing_data,
    missing_data_rate_female=variant.females_missing_data,
    y_axis_cap              =100,
    missing_data_threshold  =variant_params['miss_data_rate'],
)

Here a small dashboard with a report of the per-variant missing data rate is shown, split by sex (the Y chromosome is only present in males, so missingness is computed separately to avoid confounding). The cap on the Y-axis can be selected without re-running the whole pipeline.

In [ ]:
# Regenerate the missing-data plots alone, e.g. with a different y-axis cap
report.report_variant_qc(
    missing_data_rate_male  =variant.males_missing_data,
    missing_data_rate_female=variant.females_missing_data,
    y_axis_cap              =50,
    missing_data_threshold  =variant_params['miss_data_rate'],
)

All QC failures were already aggregated by `get_fail_variants()` inside `execute_variant_qc_pipeline()`, which wrote `fail_markers.txt` and `variant_qc_summary.tsv` to disk. Let's load them to inspect how many variants failed each check.

In [ ]:
variant_summary = pd.read_csv(variant.results_dir / 'variant_qc_summary.tsv', sep='\t')
fail_markers = pd.read_csv(variant.fails_dir / 'fail_markers.txt', header=None, names=['SNP'])

In [ ]:
variant_summary

In [ ]:
print('Unique variants failing QC:', fail_markers.shape[0])

The failing variants were already dropped by `execute_drop_variants()` inside the pipeline run above (together with the `maf`/`geno`/`hwe` thresholds). The cleaned `PLINK` files are available at the path below.

In [ ]:
clean_files = variant.clean_dir / variant.output_name
print(f'Clean PLINK files (variants failing QC removed): {clean_files}')

Some intermediate files are deleted to save space.

In [ ]:
cleanup = VariantQCCleanUp(output_path=variant.results_dir)
cleanup.clean_all()